[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_DSP/Real_Time_DSP.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Real-Time Signal Processing

The workshop [Intro to GPU Systems](../Intro_GPU/Intro_GPU.ipynb) promised: what changes when the signal *keeps coming* and every block has a **deadline**. Fixed-point arithmetic, block processing under a latency budget, and a simulated real-time pipeline with measured deadline misses — the glue between [DSP](./README.md), [OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb), and [FPGA](../Intro_FPGA/README.md).

## 1. Pre-requisites

- [Filter Design](./Filter_Design.ipynb), [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) (block processing).
- [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) — scheduling jitter is the enemy here.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
import time
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *Fixed-Point Arithmetic* (~40 min)
**Goal:** represent signals in Q-format; measure quantization noise; watch overflow bite.
**Builds on:** [Intro to C](../Intro_Programming/Intro_C.ipynb) (bits). &nbsp; **Feeds into:** Session 2 (latency budgets).

---

## 2. Numbers Without a Float Unit

💡 **Intuition.** Microcontrollers and [FPGA fabric](../Intro_FPGA/Intro_FPGA.ipynb) do integer math. **Q-format** fakes fractions with an implicit binary point: Q1.15 stores $x \in [-1, 1)$ as $\mathrm{round}(x \cdot 2^{15})$ in an int16. Each quantization adds ~uniform noise of variance $\Delta^2/12$ — *6 dB of SNR per bit* — and every multiply must be re-scaled (>> 15) or the binary point drifts. The two failure modes to respect: **quantization noise** (graceful, hissy) and **overflow** (catastrophic, wrap-around).

In [ ]:

# YOUR CODE HERE


**What just happened.** Drop 4 bits and you lose about 24 dB; drop 8 and you lose about 48. The measured column falls by **23.8 dB** then **24.5 dB** as we go 15 → 11 → 7 bits, which is the 6.02 dB-per-bit rule confirmed on a real signal. This is the number to carry around: *one bit is 6 dB*, so an audio codec at 16 bits has ~96 dB of dynamic range and a 12-bit ADC gives you ~72.

**Now the interesting part.** Every row beats the printed rule of thumb by almost exactly 2.9 dB. That is not measurement luck — it is a reminder that $\mathrm{SNR} = 6.02b + 1.76$ carries an assumption most textbooks state once and never repeat: it is for a sine that *fills the full scale* of a converter whose range spans $2^b$ steps. Our setup differs in two ways that nearly cancel:

- our sine has amplitude 0.7, not 1.0, which costs $20\log_{10}(0.7) = -3.1$ dB;
- our Q-format puts $b$ fractional bits across $[-1, 1)$, so the step is $2^{-b}$ rather than $2/2^{b}$ — a factor of two finer, worth $+6.02$ dB.

Net: $+2.9$ dB, matching the gap in every row. Redo the arithmetic with our actual amplitude and step size and the prediction becomes 95.0 / 70.9 / 46.8 dB against the measured 95.2 / 71.4 / 46.9. The theory was never wrong; we were quoting it outside its assumptions.

The lesson generalizes past this cell. A rule of thumb that disagrees with your measurement by a constant offset is almost always a units or full-scale convention mismatch, not a broken system — chase the constant before you chase the code.

In [ ]:
# A Q15 FIR filter, and the overflow trap
# CORRECT: accumulate in int32 (headroom!), shift back once
# WRONG: no headroom — products wrap in int16

# YOUR CODE HERE


**What just happened.** Same filter, same input, same Q15 format — and a factor of **twelve thousand** difference in error: `5.52e-05` with an int32 accumulator versus `0.68` without. The orange trace is unusable, and notice *how* it fails: not as added hiss but as violent excursions that appear exactly where the signal is largest, because that is where products overflow and wrap from large-positive to large-negative.

The cause is arithmetic, not DSP. Multiplying two Q15 numbers produces a Q30 result, which needs 32 bits before you shift it back down to 16 — and an FIR sums 31 of those products, so the accumulator needs headroom for the sum as well. The correct version accumulates wide and shifts *once* at the end. The broken version narrows to int16 after every multiply, and the wraparound is silent: no exception, no warning, just a wrong number that keeps flowing downstream.

This is the defining hazard of fixed-point work. A float pipeline that goes wrong usually announces itself with `nan` or `inf`; an integer pipeline hands you a plausible-looking int16 and lets you ship it. The discipline that prevents it is fixed: accumulate wide, shift late, saturate rather than wrap at the boundaries (`np.clip` here is standing in for the saturating instruction a DSP core gives you in hardware).

---
### 🕐 Session 2 of 3 — *Latency Budgets & Block Processing* (~35 min)
**Goal:** count the milliseconds: block size sets the latency floor; compute must fit inside it.
**Builds on:** Session 1; [OS workshop](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb). &nbsp; **Feeds into:** Session 3 (a real-time pipeline).

---

## 3. The Budget

💡 **Intuition.** A real-time system processes block $k$ while block $k{+}1$ records. Two laws follow. **Latency floor:** you can't output before a block fills — latency ≥ one block (plus compute, plus output buffering); small blocks = low latency. **Throughput wall:** compute per block must finish in under one block-duration — or you fall behind *forever*. Small blocks also mean more per-block overhead ([OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) syscalls, scheduling), so the budget squeezes from both sides. Every audio interface's 'buffer size' knob is exactly this dial.

In [ ]:

# YOUR CODE HERE


**What just happened.** Two things, and it is worth separating them because only one is about deadlines.

**The latency floor is pure arithmetic.** A 64-sample block at 48 kHz takes 1.33 ms to fill, and 4096 samples take 85.3 ms. No amount of compute speed changes that column — it is $B/f_s$ and nothing else. This is why low-latency audio interfaces advertise small buffers, and why a 4096-sample buffer is unusable for a musician monitoring themselves live (85 ms of delay is roughly a person standing 30 metres away) but perfectly fine for offline rendering.

**The algorithm crossover is the real find.** Direct convolution wins decisively at 64 samples (0.015 ms vs 0.133 ms — nearly 9× faster than the FFT), the two roughly tie by 256, and the FFT pulls ahead at 1024 and beyond (0.055 ms vs 0.122 ms, then 0.084 ms vs 0.468 ms). Note what the FFT column *does*: it barely moves from 64 to 4096 while direct convolution's cost climbs with block size. That is $O(N \log N)$ against $O(N \cdot M)$ made visible, and it is why the fast-convolution machinery from Foundations 1 exists.

**Be honest about the verdicts.** Every configuration passes here — no `✗ MISSES` anywhere — because a 2049-tap filter on one second of audio is not much work for a modern laptop. Don't read this table as "real-time is easy." Read it as the *shape* of the budget on a fast machine with nothing else running. Three things collapse the margin in practice: an embedded target 100× slower, a system doing many channels at once rather than one, and an operating system that takes the CPU away at the wrong moment. Session 3 measures that last one.

Keep an eye on the ratio rather than the absolute numbers. At 1024 samples, FFT convolution uses 0.055 ms of a 21.3 ms budget — about 0.3% — and that fraction, not the millisecond count, is what tells you whether the system survives a bad day.

---
### 🕐 Session 3 of 3 — *A Real-Time Pipeline, Simulated & Measured* (~40 min)
**Goal:** producer/consumer with deadlines; measure misses and jitter like an engineer.
**Builds on:** Session 2.

---

## 4. The Pipeline

Architecture (the [OS workshop's](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) producer/consumer, with a clock): an acquisition thread produces blocks on schedule; a processing thread must consume + filter each block before the next arrives. We measure the **headroom histogram** — the engineer's dashboard for 'will this survive a bad scheduling day?'

In [ ]:

# YOUR CODE HERE


**What just happened.** 300 blocks, a 21.3 ms budget each, **zero deadline misses**, with median headroom 21.07 ms and a worst case of 20.81 ms. Filtering a block cost roughly 0.25 ms — about 1% of the budget — so the consumer spends almost all its time waiting for the producer, which is exactly what a healthy real-time system looks like.

**Now read the histogram the way an engineer would.** The number that matters is not the median, it is the **worst case**: 20.81 ms of headroom, so even the slowest block finished with 98% of its budget to spare. The distribution is tight, and tightness is the property you are shopping for — a real-time system is sized by its tail, because the deadline is missed by the worst block, never the average one. A system with 15 ms of *median* headroom but a worst case of 0.2 ms is far more dangerous than this one, and no summary statistic except the minimum would tell you.

**And be suspicious of this result.** It is clean because the machine was idle and the filter is cheap; it demonstrates the measurement method, not a hard scheduling problem. Two caveats to keep:

- **The load caveat.** The margin above belongs to an unloaded machine. Re-run this cell while something heavy competes for the CPU and the histogram grows a tail stretching toward the red line — the operating system's scheduler takes your margin without asking. That tail is the real subject of this session.
- **The platform caveat.** Python's GIL and `time.sleep` granularity mean this simulates real-time behaviour rather than achieving it. Production audio uses callbacks driven by the sound card's own clock and lock-free ring buffers, precisely to avoid the jitter sources we are subject to here.

What transfers is the discipline, not the code: give every block a deadline, measure headroom at completion, and judge the system by its worst block. When the tail crosses the line and it cannot be fixed in software, the escape hatches from the conclusion are what remain — [GPU batching](../Intro_GPU/README.md) for throughput, [FPGA](../Intro_FPGA/Intro_FPGA.ipynb) for deterministic latency.

## 5. Conclusion

Six dB per bit, headroom before shifting, latency ≥ one block, compute < one block-duration, and always look at the *worst-case* headroom, not the average. When the budget can't be met on a CPU, you now know both escape hatches: [GPU batching](../Intro_GPU/README.md) (throughput, at latency cost) and [FPGA](../Intro_FPGA/Intro_FPGA.ipynb) (deterministic latency, at effort cost).

---
## Where next

- [Intro to FPGA](../Intro_FPGA/Intro_FPGA.ipynb) — the same Q15 FIR as literal hardware.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — MHz-rate streams where these budgets get serious.
- [Intro to OS](../Intro_Host_Prog/Intro_OS/Intro_OS.ipynb) — the scheduler that owns your jitter.